# Tutorial 15: Small Molecule Annotation

This notebook demonstrates how to annotate small molecules with metadata
from multiple databases using the ``MoleculeAnnotator``.

**Data sources:**

| Category | Source | What you get |
|---|---|---|
| Physicochemical | RDKit (local) | MW, LogP, QED, PAINS, Lipinski, scaffold |
| Bioactivities | ChEMBL | IC50, Ki, EC50 values + targets |
| Ontological roles | ChEBI | Antibiotic, metabolite, hormone, etc. |
| Pathways | KEGG | Metabolic pathway annotations |
| Cross-database IDs | PubChem/UniChem | ChEMBL, DrugBank, KEGG, InChIKey |
| Disease associations | PubChem | Linked diseases from bioassays |

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

## 1. Physicochemical Properties (local, no API)

These are computed instantly using RDKit -- no network calls needed.

In [ ]:
from embpy.resources import MoleculeAnnotator

ma = MoleculeAnnotator()

props = ma.get_physicochemical_properties("Cn1c(=O)c2c(ncn2C)n(C)c1=O")  # Caffeine
for k, v in props.items():
    print(f"  {k:25s}: {v}")

## 2. One-Call Annotation

Use ``annotate()`` to fetch everything at once.

In [ ]:
result = ma.annotate("aspirin", sources="all")

print(f"SMILES: {result['canonical_smiles']}")
print(f"\nPhysicochemical:")
for k in ['molecular_weight', 'logp', 'qed', 'lipinski_violations']:
    print(f"  {k}: {result['physicochemical'].get(k)}")

print(f"\nTargets: {len(result.get('targets', []))}")
for t in result.get('targets', [])[:5]:
    print(f"  {t['target_name']}")

print(f"\nCross-references: {result.get('cross_references', {})}")

## 3. Annotate an AnnData

For perturbation datasets, annotate molecules in ``.obs`` directly.

In [ ]:
import pandas as pd
from anndata import AnnData

adata = AnnData(
    obs=pd.DataFrame({"drug": ["aspirin", "caffeine", "ibuprofen", "aspirin"]}),
)
adata.obs.index = [f"cell_{i}" for i in range(4)]

result = ma.annotate_adata(adata, column="drug", sources="structural")

print(result.obs[["drug", "mol_molecular_weight", "mol_logp", "mol_qed"]].to_string())

## 4. Via the tl Module

The convenience function ``annotate_molecules()`` does the same thing.

In [ ]:
from embpy.tl import annotate_molecules

result = annotate_molecules(adata, column="drug", sources="structural")
print(result.obs.columns.tolist())

## Summary

| What | How |
|---|---|
| Physicochemical properties | ``ma.get_physicochemical_properties(smiles)`` |
| Bioactivities from ChEMBL | ``ma.get_bioactivities(smiles)`` |
| Target proteins | ``ma.get_target_proteins(smiles)`` |
| Mechanism of action | ``ma.get_mechanism_of_action(smiles)`` |
| ChEBI roles | ``ma.get_chebi_roles(name)`` |
| KEGG pathways | ``ma.get_kegg_pathways(name)`` |
| Cross-database IDs | ``ma.get_cross_references(name)`` |
| All at once | ``ma.annotate(identifier, sources='all')`` |
| Annotate AnnData | ``annotate_molecules(adata, column='drug')`` |